In [0]:
%sh pip install sodapy

In [0]:
from sodapy import Socrata
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

APP_TOKEN = dbutils.secrets.get("biz-survival", "nyc_app_token")
lake_table_path = "/Volumes/data_515/default/biz_survival/311_Service_Requests_from_2020_to_Present_20260222.csv"

try:
    df_existing = spark.read.format("csv").option("header", True).load(lake_table_path)
    
    last_created_date_str = df_existing.agg(F.max('Created Date')).collect()[0][0]
    try:
        last_created_date_dt = datetime.strptime(last_created_date_str, "%m/%d/%Y %I:%M:%S %p")
        last_created_date = last_created_date_dt.strftime("%Y-%m-%dT%H:%M:%S")
    except Exception:
        last_created_date = last_created_date_str 
except:
    last_created_date = "2026-01-01T00:00:00"

client = Socrata("data.cityofnewyork.us", APP_TOKEN)

soql_query = f"SELECT * WHERE created_date > '{last_created_date}'"
results = client.get("erm2-nwe9", query=soql_query)

if results:
    df_new_data = spark.createDataFrame(results)
#     df_new_data = df_new_data.withColumn("Created Date", F.to_timestamp("Created Date"))

#     if not DeltaTable.isDeltaTable(spark, lake_table_path):
#         df_new_data.write.format("delta").mode("overwrite").save(lake_table_path)
#         print(f"Created new table with {df_new_data.count()} records.")
#     else:
#         delta_table = DeltaTable.forPath(spark, lake_table_path)
        
#         (delta_table.alias("target")
#          .merge(
#              df_new_data.alias("updates"),
#              "target.license_number = updates.license_number" # Unique ID for upsert
#          )
#          .whenMatchedUpdateAll()
#          .whenNotMatchedInsertAll()
#          .execute())
#         print(f"Upserted {df_new_data.count()} new/updated records.")
# else:
#     print("No new records found.")

In [0]:
last_created_date

In [0]:
results

In [0]:
df_new_data = spark.createDataFrame(results)
df_new_data.display()

In [0]:
df_existing.display()